In [ ]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import os
import random
import hashlib
from datetime import date, timedelta
import warnings
warnings.filterwarnings("ignore")

fake = Faker("en_US")
np.random.seed(42)
random.seed(42)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/My Drive/Colab Notebooks/Warehouse_Order_Forecast"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory ready: {OUTPUT_DIR}")

Mounted at /content/drive
✅ Output directory ready: /content/drive/My Drive/Colab Notebooks/Warehouse_Order_Forecast


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
START_DATE    = date(2022, 1, 1)
END_DATE      = date(2025, 12, 31)
N_SKUS        = 600
N_WAREHOUSES  = 5
BINS_PER_WH   = 3_000
ORDERS_PER_YEAR_PER_WH = 50_000   # ≈ 1 M rows total
YEARS         = list(range(2022, 2026))

CATEGORIES = {
    "Braking":     0.20,
    "Electrical":  0.20,
    "Filtration":  0.15,
    "Tires":       0.20,
    "AC Parts":    0.10,
    "Suspension":  0.08,
    "Drivetrain":  0.07,
}

WAREHOUSES_INFO = [
    {"warehouse_id": "WH001", "city": "Los Angeles",  "state": "CA", "region": "West"},
    {"warehouse_id": "WH002", "city": "New York",      "state": "NY", "region": "Northeast"},
    {"warehouse_id": "WH003", "city": "Orlando",       "state": "FL", "region": "Southeast"},
    {"warehouse_id": "WH004", "city": "Kansas City",   "state": "MO", "region": "Midwest"},
    {"warehouse_id": "WH005", "city": "Phoenix",       "state": "AZ", "region": "Southwest"},
]

# Class A = top 20% SKUs → 80% of volume  (Pareto)
CLASS_A_SHARE = 0.20
CLASS_A_VOLUME_WEIGHT = 80
CLASS_B_VOLUME_WEIGHT = 20

In [ ]:
# ============================================================
# 1. dim_warehouses
# ============================================================
print("\n📦 Building dim_warehouses …")

wh_rows = []
for w in WAREHOUSES_INFO:
    wh_rows.append({
        "warehouse_id":   w["warehouse_id"],
        "warehouse_name": f"{w['city']} Hub",
        "city":           w["city"],
        "state":          w["state"],
        "region":         w["region"],
        "address":        fake.street_address(),
        "zip_code":       fake.zipcode(),
        "total_bins":     BINS_PER_WH,
        "active_flag":    True,
    })

dim_warehouses = pd.DataFrame(wh_rows)
dim_warehouses.to_csv(f"{OUTPUT_DIR}/dim_warehouses.csv", index=False)
print(f"   ✔ dim_warehouses: {len(dim_warehouses)} rows")



📦 Building dim_warehouses …
   ✔ dim_warehouses: 5 rows


In [ ]:
# ============================================================
# 2. dim_products
# ============================================================
print("\n🔧 Building dim_products …")

cat_list  = list(CATEGORIES.keys())
cat_probs = list(CATEGORIES.values())

n_class_a = int(N_SKUS * CLASS_A_SHARE)   # 120 Class A SKUs
n_class_b = N_SKUS - n_class_a            # 480 Class B SKUs

product_rows = []
sku_counter  = 1

for cls, count in [("A", n_class_a), ("B", n_class_b)]:
    for _ in range(count):
        category = random.choices(cat_list, weights=cat_probs)[0]
        sku_id   = f"SKU-{sku_counter:04d}"
        product_rows.append({
            "product_id":    sku_id,
            "product_name":  f"{fake.word().capitalize()} {category} Part {sku_counter}",
            "category":      category,
            "sub_category":  fake.word().capitalize(),
            "brand":         fake.company().split()[0],
            "unit_cost":     round(random.uniform(5, 500), 2),
            "unit_price":    round(random.uniform(10, 800), 2),
            "weight_lbs":    round(random.uniform(0.1, 50), 2),
            "lead_time_days":random.randint(1, 30),
            "reorder_point": random.randint(10, 100),
            "abc_class":     cls,
            "active_flag":   True,
        })
        sku_counter += 1

dim_products = pd.DataFrame(product_rows)
dim_products.to_csv(f"{OUTPUT_DIR}/dim_products.csv", index=False)
print(f"   ✔ dim_products: {len(dim_products)} rows  (Class A: {n_class_a}, Class B: {n_class_b})")



🔧 Building dim_products …
   ✔ dim_products: 600 rows  (Class A: 120, Class B: 480)


In [ ]:
# ============================================================
# 3. dim_calendar
# ============================================================
print("\n📅 Building dim_calendar …")

date_range = pd.date_range(START_DATE, END_DATE, freq="D")
cal_rows   = []
for d in date_range:
    cal_rows.append({
        "date_id":        d.strftime("%Y%m%d"),
        "full_date":      d.date(),
        "year":           d.year,
        "quarter":        d.quarter,
        "month":          d.month,
        "month_name":     d.strftime("%B"),
        "week_of_year":   d.isocalendar()[1],
        "day_of_week":    d.dayofweek + 1,         # 1=Mon … 7=Sun
        "day_name":       d.strftime("%A"),
        "is_weekend":     d.dayofweek >= 5,
        "is_winter":      d.month in (12, 1, 2),
        "is_summer":      d.month in (6, 7, 8),
        "fiscal_year":    d.year if d.month >= 10 else d.year - 1,
    })

dim_calendar = pd.DataFrame(cal_rows)
dim_calendar.to_csv(f"{OUTPUT_DIR}/dim_calendar.csv", index=False)
print(f"   ✔ dim_calendar: {len(dim_calendar)} rows")



📅 Building dim_calendar …
   ✔ dim_calendar: 1461 rows


In [ ]:
# ============================================================
# HELPER – seasonality multiplier
# ============================================================
def seasonal_multiplier(category: str, wh_id: str, month: int) -> float:
    """
    +20% (×1.20) for:
      • Tires  in WH002 (NY) or WH004 (Kansas) during Winter  (Dec/Jan/Feb)
      • AC Parts in WH005 (Phoenix) or WH003 (Orlando) during Summer (Jun/Jul/Aug)
    """
    if category == "Tires" and wh_id in ("WH002", "WH004") and month in (12, 1, 2):
        return 1.20
    if category == "AC Parts" and wh_id in ("WH003", "WH005") and month in (6, 7, 8):
        return 1.20
    return 1.00


# Pre-build lookup: product_id → (category, abc_class)
prod_meta = dim_products.set_index("product_id")[["category", "abc_class"]].to_dict("index")

# Pareto weights per SKU
sku_ids = dim_products["product_id"].tolist()
sku_weights = np.array([
    CLASS_A_VOLUME_WEIGHT if prod_meta[s]["abc_class"] == "A" else CLASS_B_VOLUME_WEIGHT
    for s in sku_ids
], dtype=float)
sku_weights /= sku_weights.sum()   # normalise → probabilities



In [ ]:
# ============================================================
# 4 & 5. fact_orders_header + fact_orders_detail  (year-by-year)
# ============================================================
print("\n📋 Building fact_orders_header + fact_orders_detail (year-by-year) …")

HEADER_COLS = [
    "order_id", "order_date", "date_id", "warehouse_id",
    "order_type", "priority", "status", "total_lines",
    "gross_amount", "discount_amount", "net_amount",
]
DETAIL_COLS = [
    "order_detail_id", "order_id", "product_id", "bin_id",
    "ordered_qty", "unit_price", "line_gross", "discount_pct", "line_net",
]

order_counter  = 1
detail_counter = 1
ORDER_TYPES    = ["Outbound", "Inbound", "Transfer", "Return"]
PRIORITIES     = ["High", "Medium", "Low"]
STATUSES       = ["Completed", "Pending", "Cancelled", "Backordered"]

# Bin pool per warehouse (pre-generate)
wh_bins = {
    w["warehouse_id"]: [f"{w['warehouse_id']}-BIN-{b:04d}" for b in range(1, BINS_PER_WH + 1)]
    for w in WAREHOUSES_INFO
}

for year in YEARS:
    print(f"   ⏳ Year {year} …", end=" ", flush=True)

    # Date pool for the year
    yr_dates  = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
    yr_dates_series = pd.Series(yr_dates)

    hdr_rows = []
    dtl_rows = []

    for wh in WAREHOUSES_INFO:
        wh_id     = wh["warehouse_id"]
        bin_pool  = wh_bins[wh_id]

        # Build a date array biased by daily load (uniform for simplicity)
        order_dates = np.random.choice(yr_dates, size=ORDERS_PER_YEAR_PER_WH, replace=True)
        order_dates_sorted = np.sort(order_dates)

        for i in range(ORDERS_PER_YEAR_PER_WH):
            od        = pd.Timestamp(order_dates_sorted[i])
            month     = od.month
            date_id   = od.strftime("%Y%m%d")
            order_id  = f"ORD-{order_counter:08d}"
            order_counter += 1

            n_lines    = random.randint(1, 5)

            # Sample SKUs with Pareto weights + apply seasonality boost
            month_weights = sku_weights.copy()
            for idx, sk in enumerate(sku_ids):
                mult = seasonal_multiplier(prod_meta[sk]["category"], wh_id, month)
                month_weights[idx] *= mult
            month_weights /= month_weights.sum()

            chosen_skus = np.random.choice(sku_ids, size=n_lines, replace=False, p=month_weights)

            gross_total   = 0.0
            disc_total    = 0.0

            for j, sku in enumerate(chosen_skus):
                qty        = random.randint(1, 50)
                price      = dim_products.loc[dim_products["product_id"] == sku, "unit_price"].values[0]
                disc_pct   = round(random.uniform(0, 0.25), 4)
                line_gross = round(qty * price, 2)
                line_net   = round(line_gross * (1 - disc_pct), 2)
                bin_id     = random.choice(bin_pool)

                dtl_rows.append({
                    "order_detail_id": f"DTL-{detail_counter:010d}",
                    "order_id":        order_id,
                    "product_id":      sku,
                    "bin_id":          bin_id,
                    "ordered_qty":     qty,
                    "unit_price":      price,
                    "line_gross":      line_gross,
                    "discount_pct":    disc_pct,
                    "line_net":        line_net,
                })
                detail_counter += 1
                gross_total += line_gross
                disc_total  += round(line_gross * disc_pct, 2)

            hdr_rows.append({
                "order_id":        order_id,
                "order_date":      od.date(),
                "date_id":         date_id,
                "warehouse_id":    wh_id,
                "order_type":      random.choice(ORDER_TYPES),
                "priority":        random.choice(PRIORITIES),
                "status":          random.choices(STATUSES, weights=[70, 15, 5, 10])[0],
                "total_lines":     n_lines,
                "gross_amount":    round(gross_total, 2),
                "discount_amount": round(disc_total, 2),
                "net_amount":      round(gross_total - disc_total, 2),
            })

    # Write year chunk
    mode = "w" if year == YEARS[0] else "a"
    hdr_df = pd.DataFrame(hdr_rows, columns=HEADER_COLS)
    dtl_df = pd.DataFrame(dtl_rows, columns=DETAIL_COLS)

    hdr_df.to_csv(f"{OUTPUT_DIR}/fact_orders_header.csv", mode=mode,
                  index=False, header=(year == YEARS[0]))
    dtl_df.to_csv(f"{OUTPUT_DIR}/fact_orders_detail.csv", mode=mode,
                  index=False, header=(year == YEARS[0]))

    print(f"Orders: {len(hdr_rows):,}  |  Detail lines: {len(dtl_rows):,}")

print("   ✔ fact_orders_header & fact_orders_detail written (all years).")



📋 Building fact_orders_header + fact_orders_detail (year-by-year) …
   ⏳ Year 2022 … Orders: 250,000  |  Detail lines: 749,363
   ⏳ Year 2023 … Orders: 250,000  |  Detail lines: 750,156
   ⏳ Year 2024 … Orders: 250,000  |  Detail lines: 750,012
   ⏳ Year 2025 … Orders: 250,000  |  Detail lines: 750,533
   ✔ fact_orders_header & fact_orders_detail written (all years).


In [ ]:
# ============================================================
# 6. fact_inventory  (one snapshot per warehouse × SKU)
# ============================================================
print("\n🏗 Building fact_inventory …")

SNAPSHOT_DATE = END_DATE          # latest date = end of 2025
STOCKOUT_RATE = 0.08              # ~8% of records have available_qty = 0

inv_rows = []
for wh in WAREHOUSES_INFO:
    wh_id = wh["warehouse_id"]
    for _, prod in dim_products.iterrows():
        is_stockout  = random.random() < STOCKOUT_RATE
        on_hand      = 0 if is_stockout else random.randint(0, 500)
        allocated    = on_hand if is_stockout else random.randint(0, on_hand)
        available    = on_hand - allocated          # can be 0 for stockouts
        in_transit   = random.randint(0, 100)
        reorder_flag = available <= prod["reorder_point"]

        inv_rows.append({
            "inventory_id":   f"{wh_id}-{prod['product_id']}",
            "snapshot_date":  SNAPSHOT_DATE,
            "warehouse_id":   wh_id,
            "product_id":     prod["product_id"],
            "bin_id":         random.choice(wh_bins[wh_id]),
            "on_hand":        on_hand,
            "allocated":      allocated,
            "available_qty":  available,
            "in_transit":     in_transit,
            "reorder_flag":   reorder_flag,
            "abc_class":      prod["abc_class"],
            "last_movement":  fake.date_between(start_date="-90d", end_date="today"),
        })

fact_inventory = pd.DataFrame(inv_rows)
fact_inventory.to_csv(f"{OUTPUT_DIR}/fact_inventory.csv", index=False)
stockout_count = (fact_inventory["available_qty"] == 0).sum()
print(f"   ✔ fact_inventory: {len(fact_inventory):,} rows  |  Stockouts: {stockout_count:,} ({stockout_count/len(fact_inventory)*100:.1f}%)")



🏗 Building fact_inventory …
   ✔ fact_inventory: 3,000 rows  |  Stockouts: 300 (10.0%)


In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("✅  DATASET GENERATION COMPLETE")
print("="*60)
files = [
    "dim_warehouses.csv", "dim_products.csv", "dim_calendar.csv",
    "fact_orders_header.csv", "fact_orders_detail.csv", "fact_inventory.csv"
]
for f in files:
    path = f"{OUTPUT_DIR}/{f}"
    size_mb = os.path.getsize(path) / 1_048_576
    row_count = sum(1 for _ in open(path)) - 1   # subtract header
    print(f"   {f:<35} {row_count:>10,} rows   {size_mb:>7.1f} MB")

print(f"\n📁 All files saved to:\n   {OUTPUT_DIR}")


✅  DATASET GENERATION COMPLETE
   dim_warehouses.csv                           5 rows       0.0 MB
   dim_products.csv                           600 rows       0.1 MB
   dim_calendar.csv                         1,461 rows       0.1 MB
   fact_orders_header.csv               1,000,000 rows      85.4 MB
   fact_orders_detail.csv               3,000,064 rows     242.5 MB
   fact_inventory.csv                       3,000 rows       0.3 MB

📁 All files saved to:
   /content/drive/My Drive/Colab Notebooks/Warehouse_Order_Forecast


In [ ]:
import pandas as pd
import os

INPUT_DIR  = "/content/drive/My Drive/Colab Notebooks/Warehouse_Order_Forecast"
INPUT_FILE = f"{INPUT_DIR}/fact_orders_detail.csv"

print("📂 Reading fact_orders_detail.csv …")
print("   (242 MB — this may take ~30–60 seconds)")

# ── Read in chunks to avoid memory pressure ─────────────────
CHUNK_SIZE = 500_000

# We need the order_date from the header to derive year.
# fact_orders_detail has order_id → join with header, OR
# we can extract year directly from order_id prefix via header lookup.
# Simplest: read the header file for the date mapping, then merge on order_id.

print("📂 Reading fact_orders_header.csv for date mapping …")
header_dates = pd.read_csv(
    f"{INPUT_DIR}/fact_orders_header.csv",
    usecols=["order_id", "order_date"],
    parse_dates=["order_date"],
)
header_dates["year"] = header_dates["order_date"].dt.year
date_map = header_dates.set_index("order_id")["year"].to_dict()
print(f"   ✔ Loaded {len(date_map):,} order → year mappings")

# ── Open one output file handle per year ────────────────────
YEARS        = [2022, 2023, 2024, 2025]
file_handles = {}
writers      = {}

for yr in YEARS:
    out_path = f"{INPUT_DIR}/fact_orders_detail_{yr}.csv"
    file_handles[yr] = open(out_path, "w", newline="", encoding="utf-8")
    writers[yr] = None   # will hold True after header written

print("\n✂️  Splitting by year (chunked) …")

chunk_num  = 0
row_counts = {yr: 0 for yr in YEARS}

for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):
    chunk_num += 1
    chunk["year"] = chunk["order_id"].map(date_map)

    for yr in YEARS:
        subset = chunk[chunk["year"] == yr].drop(columns=["year"])
        if subset.empty:
            continue

        write_header = (writers[yr] is None)   # first time for this year
        subset.to_csv(file_handles[yr], index=False, header=write_header)
        writers[yr]    = True
        row_counts[yr] += len(subset)

    total_so_far = sum(row_counts.values())
    print(f"   Chunk {chunk_num:>2} processed — cumulative rows written: {total_so_far:>10,}")

# ── Close all file handles ───────────────────────────────────
for fh in file_handles.values():
    fh.close()

# ── Summary ─────────────────────────────────────────────────
print("\n" + "="*60)
print("✅  SPLIT COMPLETE")
print("="*60)
for yr in YEARS:
    path     = f"{INPUT_DIR}/fact_orders_detail_{yr}.csv"
    size_mb  = os.path.getsize(path) / 1_048_576
    print(f"   fact_orders_detail_{yr}.csv   {row_counts[yr]:>10,} rows   {size_mb:>7.1f} MB")

print(f"\n📁 Files saved to:\n   {INPUT_DIR}")


📂 Reading fact_orders_detail.csv …
   (242 MB — this may take ~30–60 seconds)
📂 Reading fact_orders_header.csv for date mapping …
   ✔ Loaded 1,000,000 order → year mappings

✂️  Splitting by year (chunked) …
   Chunk  1 processed — cumulative rows written:    500,000
   Chunk  2 processed — cumulative rows written:  1,000,000
   Chunk  3 processed — cumulative rows written:  1,500,000
   Chunk  4 processed — cumulative rows written:  2,000,000
   Chunk  5 processed — cumulative rows written:  2,500,000
   Chunk  6 processed — cumulative rows written:  3,000,000
   Chunk  7 processed — cumulative rows written:  3,000,064

✅  SPLIT COMPLETE
   fact_orders_detail_2022.csv      749,363 rows      60.6 MB
   fact_orders_detail_2023.csv      750,156 rows      60.6 MB
   fact_orders_detail_2024.csv      750,012 rows      60.6 MB
   fact_orders_detail_2025.csv      750,533 rows      60.7 MB

📁 Files saved to:
   /content/drive/My Drive/Colab Notebooks/Warehouse_Order_Forecast
